# 01 Score Annotation

Turns the completed annotation sheets into the numbers that go in the paper:
accuracy with clustered confidence intervals, a test of whether the pipeline is
equally reliable on the two subcorpora, the coordination check, and the
error-impact table.

**Input.** `annotation_records.csv` — long format, one row per annotated unit,
labels but no word forms, so it can be published:

| Column | Meaning |
|---|---|
| `passage_id`, `text_id`, `origin_type`, `genre` | where the unit comes from |
| `task` | `boundary`, `token`, `pos`, `coord_any`, `coord_clause`, `morph_*` |
| `unit_id` | sentence or token index within the passage |
| `gold_label` | what the annotator decided |
| `pred_label` | what the pipeline produced — empty for `coord_clause` |
| `is_onomatopoeia`, `is_diminutive` | flags for the hard cases of 0+ material |

The last cell of this notebook builds that file from the annotated workbook, so
run it once before the rest if you are coming straight from annotation.

The sample is annotated by one person, so there is no inter-annotator
agreement to report. Say so explicitly in the paper rather than leaving it to
be noticed.


In [ ]:
!pip install scipy --quiet

## Parameters

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact, chi2_contingency

RECORDS  = "annotation_records.csv"
COUNTS   = "passage_counts.csv"      # optional, for the error-impact section
RESULTS  = "validation_results.csv"

SEED    = 20260807
N_BOOT  = 5000
ALPHA   = .05

rng = np.random.default_rng(SEED)

In [ ]:
df = pd.read_csv(RECORDS)

print(df.shape)
df.groupby(["task", "origin_type"]).size().unstack(fill_value=0)

## Helpers

Confidence intervals are bootstrapped by resampling **texts**, not rows.
Tokens inside one passage are not independent — a single missed sentence
boundary corrupts every token in it — so resampling rows would produce an
interval that is too narrow. This is the same objection Reviewer #3 raised
about the analysis itself, and it applies here too.


In [ ]:
def clustered_ci(d, cluster="text_id", n_boot=N_BOOT, alpha=ALPHA):
    groups = [g["correct"].values for _, g in d.groupby(cluster)]
    if len(groups) < 2:
        return (np.nan, np.nan)
    idx = np.arange(len(groups))
    means = np.empty(n_boot)
    for b in range(n_boot):
        pick = rng.choice(idx, size=len(idx), replace=True)
        means[b] = np.concatenate([groups[i] for i in pick]).mean()
    return tuple(np.quantile(means, [alpha / 2, 1 - alpha / 2]))

## Accuracy per task

`p_symmetry` tests whether accuracy differs between RUS-O and RUS-T.

**A large p is the result you want.** It means the pipeline is equally reliable
on both subcorpora, and therefore that the contrasts reported in Section 5
cannot be an artefact of unequal annotation quality. A small p would mean the
opposite and would have to be reported as a threat to the findings.


In [ ]:
def score_task(d_all, task):
    d = d_all[d_all.task == task].copy()
    if d.empty or d.pred_label.isna().all():
        return None
    d["correct"] = (d.gold_label.astype(str) == d.pred_label.astype(str)).astype(int)

    out = {"task": task, "n_units": len(d), "n_texts": d.text_id.nunique(),
           "accuracy": d.correct.mean()}
    out["ci_low"], out["ci_high"] = clustered_ci(d)

    for grp in ("RUS-O", "RUS-T"):
        key = grp.lower().replace("-", "_")
        sub = d[d.origin_type == grp]
        out[f"acc_{key}"] = sub.correct.mean() if len(sub) else np.nan
        out[f"n_{key}"]   = len(sub)

    tab = pd.crosstab(d.origin_type, d.correct)
    if tab.shape == (2, 2):
        if tab.values.min() < 5:
            out["p_symmetry"], out["symmetry_test"] = fisher_exact(tab.values)[1], "Fisher"
        else:
            out["p_symmetry"], out["symmetry_test"] = chi2_contingency(tab.values)[1], "chi2"
    else:
        out["p_symmetry"], out["symmetry_test"] = np.nan, ""

    for flag in ("is_onomatopoeia", "is_diminutive"):
        if flag in d.columns:
            sub = d[d[flag] == 1]
            out[f"acc_{flag[3:]}"] = sub.correct.mean() if len(sub) else np.nan
            out[f"n_{flag[3:]}"]   = len(sub)
    return out


results = [r for r in (score_task(df, t) for t in df.task.unique()) if r]
res = pd.DataFrame(results)

res[[c for c in ["task", "n_units", "accuracy", "ci_low", "ci_high",
                 "acc_rus_o", "acc_rus_t", "p_symmetry"] if c in res.columns]].round(4)

### Hard cases

Accuracy on onomatopoeia and diminutives specifically. These are the phenomena
Reviewer #3 named, and they are where a model trained on SynTagRus is most
likely to fail. Report them as a separate line even — especially — if accuracy
there is low: an honest low number with a bound on its influence is stronger
than an aggregate that hides it.


In [ ]:
cols = [c for c in ["task", "acc_onomatopoeia", "n_onomatopoeia",
                    "acc_diminutive", "n_diminutive"] if c in res.columns]
res[cols].round(4) if len(cols) > 1 else "no hard-case flags in the records"

### Confusion matrix for POS

Shows *where* the tagger fails, not just how often. Rows are gold tags, columns
are predictions.


In [ ]:
pos = df[df.task == "pos"]
if len(pos):
    cm = pd.crosstab(pos.gold_label, pos.pred_label, margins=True)
    display(cm)
    err = pos[pos.gold_label != pos.pred_label]
    print("\nmost frequent confusions:")
    print(err.groupby(["gold_label", "pred_label"]).size()
             .sort_values(ascending=False).head(10).to_string())

## Coordination: what the metric actually measures

The pipeline flags a sentence when any token carries the dependency label
`conj` or `cc`, which includes phrasal coordination — *mama i papa* counts.
The paper describes the metric as clause coordination. This section measures
the gap.

`gap_any` is what the paper currently reports: 0.091 on the full corpus. If
`gap_clause` is close to it, the effect survives and the metric only needs
renaming. If it collapses, Section 5.3 has to be rewritten.


In [ ]:
a = df[df.task == "coord_any"][["passage_id", "text_id", "origin_type",
                                "unit_id", "gold_label", "pred_label"]]
k = df[df.task == "coord_clause"][["passage_id", "unit_id", "gold_label"]]

if len(a) and len(k):
    m = a.merge(k, on=["passage_id", "unit_id"], suffixes=("_any", "_clause"))
    m = m.astype({"gold_label_any": float, "gold_label_clause": float,
                  "pred_label": float})

    coord = {
        "n_sentences": len(m),
        "gold_coord_any_rate": m.gold_label_any.mean(),
        "gold_coord_clause_rate": m.gold_label_clause.mean(),
        "pipeline_rate": m.pred_label.mean(),
        "clause_share_of_flagged": m[m.pred_label == 1].gold_label_clause.mean(),
    }
    for grp in ("RUS-O", "RUS-T"):
        s = m[m.origin_type == grp]
        coord[f"any_{grp}"] = s.gold_label_any.mean()
        coord[f"clause_{grp}"] = s.gold_label_clause.mean()
    coord["gap_any"]    = coord["any_RUS-O"] - coord["any_RUS-T"]
    coord["gap_clause"] = coord["clause_RUS-O"] - coord["clause_RUS-T"]

    pd.Series(coord).round(4).to_frame("value")

## Verification bias check

Part-of-speech accuracy is measured twice on the same 100 tokens: once cold
(`pos_blind`) and once with the model's tag on screen (`pos`). Verification is
faster but biased upwards — a plausible wrong tag gets accepted.

If the two agree within noise, report the verification figure over all ~320
tokens and cite this check. If verification is clearly higher, report the blind
figure and say why.

`n_discordant` is the number of tokens where the two passes disagree. With
around 100 tokens and a low error rate this is a handful of items: the check
catches gross acquiescence, not a two-point difference. Say so in the paper
rather than overselling it.


In [ ]:
pv = df[df.task == "pos"].set_index(["passage_id", "unit_id"])
pb = df[df.task == "pos_blind"].set_index(["passage_id", "unit_id"])
both = pb.index.intersection(pv.index)

if len(both):
    v = pv.loc[both]
    b = pb.loc[both]
    ok_v = (v.gold_label.astype(str) == v.pred_label.astype(str))
    ok_b = (b.gold_label.astype(str) == b.pred_label.astype(str))

    check = pd.Series({
        "n_tokens_compared": len(both),
        "accuracy_verification": ok_v.mean(),
        "accuracy_blind": ok_b.mean(),
        "difference": ok_v.mean() - ok_b.mean(),
        "n_discordant": int((ok_v.values != ok_b.values).sum()),
        "accuracy_verification_full": (
            df[df.task == "pos"].pipe(
                lambda d: (d.gold_label.astype(str) == d.pred_label.astype(str)).mean())),
        "n_tokens_full": int((df.task == "pos").sum()),
    })
    display(check.round(4).to_frame("value"))

    if abs(check.difference) < .03:
        print("\nThe two passes agree. Report the verification figure over all "
              f"{int(check.n_tokens_full)} tokens and cite this check.")
    else:
        print("\nVerification and blind annotation diverge. Report the blind "
              "figure and describe the discrepancy.")
else:
    print("no overlapping tokens — check that C_pos_blind is a subset of C_pos_verify")

## Error impact — the part that answers the reviewer

Accuracy on its own settles nothing: is 95% good or bad? The answer comes from
comparing the size of the tool's error to the size of the effect being claimed.

Supply `passage_counts.csv` with, for each passage, the sentence and token
counts under both the pipeline and the gold annotation:

`passage_id, text_id, origin_type, n_sent_pred, n_sent_gold, n_tok_pred, n_tok_gold`

The claim to be established is twofold. For the **significant** effects, the
instrumental error must be an order of magnitude smaller than the between-group
difference. For the **null** results — the headline contribution of the paper —
the systematic deviation must correspond to an effect size far below what the
sample is powered to detect (|d| = 0.5 at power 0.85), which rules out the
objection that the nulls are measurement noise.


In [ ]:
import os

if os.path.exists(COUNTS):
    pc = pd.read_csv(COUNTS)
    pc["asw_pred"] = pc.n_tok_pred / pc.n_sent_pred
    pc["asw_gold"] = pc.n_tok_gold / pc.n_sent_gold

    impact = []
    for label, pred, gold in [
        ("sentences per passage", "n_sent_pred", "n_sent_gold"),
        ("tokens per passage",    "n_tok_pred",  "n_tok_gold"),
        ("avg sentence length (words)", "asw_pred", "asw_gold"),
    ]:
        delta = pc[pred] - pc[gold]
        pooled = np.sqrt((pc[pred].std(ddof=1) ** 2 + pc[gold].std(ddof=1) ** 2) / 2)
        impact.append(dict(metric=label,
                           mean_pipeline=pc[pred].mean(),
                           mean_gold=pc[gold].mean(),
                           mean_delta=delta.mean(),
                           abs_delta=delta.abs().mean(),
                           delta_as_d=delta.mean() / pooled if pooled else np.nan))
    impact = pd.DataFrame(impact)
    display(impact.round(4))
    print("\ndelta_as_d is the systematic shift expressed as an effect size.")
    print("Compare it with the |d| = 0.5 the study is powered to detect and with")
    print("the effect sizes in results_main.csv.")
else:
    print(f"{COUNTS} not found — fill it in after annotation, then rerun.")

In [ ]:
# between-group differences for the same metrics, for the comparison column
main = pd.read_csv("../data/results_main.csv")
main[main.metric_id.isin(["n_sentences", "n_words", "avg_sent_words",
                          "coord_sent_ratio"])][
    ["metric_label", "mean_rus_o", "mean_rus_t", "cohens_d", "p_adj_bh"]]

## Write the results

In [ ]:
res.to_csv(RESULTS, index=False, encoding="utf-8", lineterminator="\n")
print("written:", RESULTS)

---

## Appendix — building `annotation_records.csv` from the workbook

Run this once, in the private working directory, after annotation is finished.
It reads the three exported tabs, reshapes them into the long publishable
format, and drops every word form.

Two things happen here that are worth understanding.

**Tokenisation is expanded back to token level.** You judged whole sentences;
this turns each sentence into one row per token, correct everywhere except the
positions you listed in `bad_token_positions`. The resulting accuracy is a
genuine token-level figure, not an approximation — a sentence marked
`all_correct = 1` asserts that every one of its tokens is right.

**`passage_counts.csv` is derived, not typed.** Gold sentence counts come from
the boundary errors (a `missed` adds a sentence, a `spurious` removes one) and
gold token counts from `n_tokens_gold`. That file feeds the error-impact
section above.

Check the output for stray text before committing it.


In [ ]:
WORK_DIR = "./private_sheets"


def build_records(work_dir=WORK_DIR, out="annotation_records.csv",
                  counts_out="passage_counts.csv"):
    """Reshape the three annotated tabs into the long publishable format."""
    keep = ["passage_id", "text_id", "origin_type", "genre"]

    ad = pd.read_csv(f"{work_dir}/A_D_sentences.csv")
    bt = pd.read_csv(f"{work_dir}/B_tokenisation.csv")
    cv = pd.read_csv(f"{work_dir}/C_pos_verify.csv")
    cb = pd.read_csv(f"{work_dir}/C_pos_blind.csv")

    for name, frame, cols in [("A_D_sentences", ad, ["boundary_ok", "coord_any",
                                                     "coord_clause"]),
                              ("B_tokenisation", bt, ["all_correct"]),
                              ("C_pos_verify", cv, ["tag_ok"]),
                              ("C_pos_blind", cb, ["gold_pos"])]:
        missing = frame[cols].isna().any(axis=1).sum()
        if missing:
            print(f"  warning: {missing} unfilled rows in {name}")

    parts = []

    # --- A and D: one row per sentence -------------------------------------
    ad["unit_id"] = "s" + ad.sent_index.astype(str)
    for task, col in [("boundary", "boundary_ok"),
                      ("coord_any", "coord_any"),
                      ("coord_clause", "coord_clause")]:
        p = ad[keep + ["unit_id"]].copy()
        p["task"] = task
        p["gold_label"] = ad[col].values
        parts.append(p)

    # --- B: expand the sentence-level judgement to token level --------------
    genre = ad.set_index(["passage_id", "sent_index"]).genre.to_dict()
    rows = []
    for r in bt.itertuples(index=False):
        bad = set()
        if r.all_correct == 0 and isinstance(r.bad_token_positions, str):
            bad = {int(x) for x in str(r.bad_token_positions).replace(" ", "").split(",") if x != ""}
        for j in range(int(r.n_tokens)):
            rows.append(dict(passage_id=r.passage_id, text_id=r.text_id,
                             origin_type=r.origin_type,
                             genre=genre.get((r.passage_id, r.sent_index)),
                             unit_id=f"s{r.sent_index}t{j}", task="token",
                             gold_label=int(j not in bad)))
    parts.append(pd.DataFrame(rows))

    # --- C: verification pass ----------------------------------------------
    # tag_ok = 1 means the prediction was right, so gold == pred there;
    # otherwise gold is what the annotator wrote in correct_pos.
    for frame in (cv, cb):
        frame["unit_id"] = ("s" + frame.sent_index.astype(str)
                            + "t" + frame.token_index.astype(str))
        frame["genre"] = [genre.get((p_, s_))
                          for p_, s_ in zip(frame.passage_id, frame.sent_index)]

    p = cv[keep + ["unit_id", "is_onomatopoeia", "is_diminutive"]].copy()
    p["task"] = "pos"
    p["gold_label"] = np.where(cv.tag_ok == 1, cv.pred_pos, cv.correct_pos)
    p["pred_label"] = cv.pred_pos.values
    parts.append(p)

    # --- C: blind pass, the bias check -------------------------------------
    pred_by_unit = cv.set_index(["passage_id", "unit_id"]).pred_pos
    p = cb[keep + ["unit_id"]].copy()
    p["task"] = "pos_blind"
    p["gold_label"] = cb.gold_pos.values
    p["pred_label"] = [pred_by_unit.get((a, b), np.nan)
                       for a, b in zip(cb.passage_id, cb.unit_id)]
    parts.append(p)

    rec = pd.concat(parts, ignore_index=True)
    rec["pred_label"] = rec.pred_label.where(
        rec.task.isin(["pos", "pos_blind"]), np.nan)   # others filled below
    for flag in ("is_onomatopoeia", "is_diminutive"):
        rec[flag] = rec[flag].fillna(0).astype(int)

    forbidden = {"token", "sentence", "context", "tokenisation", "text"}
    assert not (set(rec.columns) & forbidden), "word forms leaked into the records"
    rec.to_csv(out, index=False, encoding="utf-8", lineterminator="\n")

    # --- passage-level counts for the error-impact section ------------------
    err = ad.groupby("passage_id").error_type.value_counts().unstack(fill_value=0)
    for col in ("missed", "spurious"):
        if col not in err:
            err[col] = 0
    counts = (bt.groupby(["passage_id", "text_id", "origin_type"])
                .agg(n_sent_pred=("sent_index", "count"),
                     n_tok_pred=("n_tokens", "sum"),
                     n_tok_gold=("n_tokens_gold", "sum"))
                .reset_index())
    counts["n_sent_gold"] = (counts.n_sent_pred
                             + counts.passage_id.map(err["missed"]).fillna(0)
                             - counts.passage_id.map(err["spurious"]).fillna(0))
    counts.to_csv(counts_out, index=False, encoding="utf-8", lineterminator="\n")

    print(f"{rec.shape} -> {out}")
    print(f"{counts.shape} -> {counts_out}")
    return rec


# rec = build_records()
# then fill `pred_label` from the pipeline output and rerun this notebook
